# Конспект курса MCP от Hugging Face (февраль 2026)

## Введение в MCP

**MCP (Model Context Protocol)** — это открытый протокол, который позволяет AI-моделям (например, большим языковым моделям) безопасно взаимодействовать с внешними источниками данных, инструментами и окружениями. Благодаря MCP модели могут выполнять действия, получать актуальную информацию и работать с пользовательскими сервисами, не требуя жесткой интеграции каждого инструмента отдельно.

Протокол разработан для унификации подключения клиентов (хостов) к серверам, предоставляющим функциональность. MCP активно поддерживается сообществом и используется в таких продуктах, как Claude Desktop, VS Code и инструментах Hugging Face.

---

## Ключевые понятия

- **Host** — приложение на стороне пользователя (например, Claude Desktop, IDE), которое инициирует подключение к MCP-серверу и управляет взаимодействием с пользователем.
- **Client** — компонент внутри Host, отвечающий за установление соединения и обмен сообщениями с внешним MCP-сервером.
- **Server** — внешняя программа (локальная или удалённая), предоставляющая через MCP свои возможности:
  - **Tools** — функции, которые модель может вызывать (например, поиск в интернете, расчёты).
  - **Resources** — данные только для чтения (файлы, содержимое API, документы).
  - **Prompts** — готовые инструкции или шаблоны для модели.

---

## Архитектурные компоненты

```
+-------------+          +--------------+          +----------------+
|   Host      |          |   Client     |          |    Server      |
| (User App)  | <------> | (MCP Client) | <------> | (MCP Server)   |
+-------------+          +--------------+          +----------------+
```

1. Пользователь взаимодействует с Host-приложением.
2. Host внутри себя создаёт один или несколько экземпляров **Client**.
3. Каждый Client подключается к конкретному **Server** (1:1 отношение).
4. Server выполняет запросы Client и возвращает результаты.
5. Host интегрирует полученные данные и показывает их пользователю.

Такой подход позволяет легко подключать множество независимых серверов к одному приложению.

---

## Протокол взаимодействия

MCP использует **JSON-RPC 2.0** для обмена сообщениями между Client и Server.

### Типы сообщений

- **Request** (Client → Server) — инициирует операцию. Содержит уникальный `id`, имя метода и параметры.
- **Response** (Server → Client) — ответ на конкретный запрос. Содержит тот же `id` и либо `result`, либо `error`.
- **Notification** (Client → Server или Server → Client) — уведомление о событии, не требует ответа (нет `id`).

### Транспорт

MCP поддерживает два основных транспорта:

1. **stdio** — общение через стандартные потоки ввода/вывода. Используется, когда клиент и сервер запущены на одной машине. Обеспечивает изоляцию и безопасность (нет сетевого взаимодействия).
2. **HTTP + SSE (Server-Sent Events)** — для удалённого доступа. Клиент устанавливает соединение через HTTP, сервер использует SSE для отправки обновлений.

---

## Жизненный цикл интеграции

1. **Инициализация**  
   Client подключается к Server, обмениваются метаданными (версия протокола, поддерживаемые возможности). Client подтверждает инициализацию notification-сообщением.

2. **Запрос возможностей**  
   Client опрашивает Server о доступных инструментах, ресурсах и промптах.

3. **Выполнение операции**  
   На основе запроса от Host Client вызывает нужный метод на Server (например, вызывает tool). Server возвращает результат.

4. **Завершение**  
   Client отправляет notification о завершении работы, соединение закрывается.

---

## Возможности MCP Server

Server может реализовывать одну или несколько категорий возможностей:

### Tools (инструменты)
Функции, которые модель может вызывать для выполнения действий.  
*Пример:* tool для получения погоды, для выполнения кода, для поиска в базе данных.

### Resources (ресурсы)
Данные только для чтения, доступные модели.  
*Пример:* файл конфигурации, содержимое веб-страницы, результаты запроса к API.

### Prompts (промпты)
Готовые инструкции или шаблоны, которые помогают модели правильно обработать запрос пользователя.  
*Пример:* промпт для перевода текста, для генерации отчёта.

### Sampling (выборка)
Позволяет серверу запрашивать у клиента (Host) выполнение LLM-взаимодействия.  
- Server отправляет запрос на семплинг.  
- Клиент (Host) проверяет и выполняет запрос через свою модель.  
- Результат возвращается на Server.

Все категории могут комбинироваться в одном сервере, предоставляя комплексную функциональность.

---

## MCP SDK

Для упрощения разработки MCP-серверов и клиентов существуют SDK для Python и JavaScript.

### Python SDK (FastMCP)

Установка:
```bash
pip install mcp
```

Пример создания простого сервера с инструментом и ресурсом:

```python
# server.py
from mcp.server.fastmcp import FastMCP

# Создаём экземпляр сервера
mcp = FastMCP("Мой первый MCP сервер")

# Определяем инструмент
@mcp.tool()
def calculate(expression: str) -> str:
    """Вычисляет математическое выражение (например, '2+2')"""
    try:
        result = eval(expression)
        return f"Результат: {result}"
    except Exception as e:
        return f"Ошибка: {e}"

# Определяем ресурс (доступен по URI)
@mcp.resource("config://app")
def get_config() -> str:
    """Возвращает конфигурацию приложения"""
    return "version: 1.0\nmode: production"

# Определяем промпт
@mcp.prompt()
def translate_prompt(text: str, language: str = "русский") -> str:
    """Шаблон для перевода текста"""
    return f"Переведи следующий текст на {language}:\n{text}"

# Запускаем сервер (транспорт stdio по умолчанию)
if __name__ == "__main__":
    mcp.run()
```

Запуск сервера:
```bash
python server.py
```

### MCP Inspector

Для отладки и тестирования сервера можно использовать MCP Inspector — веб-интерфейс, который подключается к вашему серверу и позволяет просматривать доступные методы, вызывать их и видеть результаты.

Запуск инспектора:
```bash
npx @modelcontextprotocol/inspector python server.py
```

После запуска откроется браузер с URL, указанным в логах (обычно `http://localhost:5173`).

### Режим разработки

Для быстрой итерации можно использовать команду `mcp dev`, которая автоматически перезапускает сервер при изменениях кода:

```bash
mcp dev server.py
```

---

## MCP Clients (Клиенты)

Client — это мост между Host и Server. Существуют готовые клиенты, встроенные в популярные приложения.

### Пользовательские клиенты

- **Claude Desktop** — настольное приложение от Anthropic.
- **VS Code extension** — расширение для Visual Studio Code.
- **Tiny Agents** — инструмент командной строки для запуска агентов.

### Конфигурация клиентов

Каждый Host управляет подключениями к серверам через конфигурационные файлы. Обычно это JSON-файл `mcp.json` или аналогичный.

#### Пример для транспорта stdio

```json
{
  "mcpServers": {
    "my-python-server": {
      "command": "python",
      "args": ["/path/to/server.py"],
      "env": {
        "API_KEY": "your-key-here"
      }
    }
  }
}
```

#### Пример для транспорта SSE (удалённый сервер)

```json
{
  "mcpServers": {
    "remote-server": {
      "url": "https://example.com/mcp",
      "headers": {
        "Authorization": "Bearer YOUR_TOKEN"
      }
    }
  }
}
```

#### Переменные окружения

Чтобы передать секреты, можно:
1. Загрузить переменные в коде сервера через `load_dotenv()`.
2. Передать их в конфигурации клиента в поле `env`.
3. Для удалённых серверов использовать заголовки (например, `Authorization`).

---

## Tiny Agents Client

**Tiny Agents** — это легковесный CLI-инструмент для запуска агентов, использующих MCP-серверы.

Установка:
```bash
npm install -g tiny-agents
pip install huggingface-hub
huggingface-cli login
```

Создайте файл `agent.json`:

```json
{
  "name": "Мой агент",
  "description": "Агент для работы с калькулятором",
  "model": "gpt-4",
  "providers": ["openai"],
  "servers": [
    {
      "type": "stdio",
      "command": "python",
      "args": ["server.py"]
    }
  ]
}
```

Запуск:
```bash
tiny-agents run agent.json
```

После запуска откроется интерактивный CLI, где можно вводить запросы, и агент будет использовать доступные инструменты из сервера для выполнения задач.

---

## Hugging Face MCP Server

Hugging Face предоставляет готовый MCP-сервер, который открывает доступ к Hugging Face Hub. С его помощью можно:

- Искать модели, датасеты, Space.
- Получать информацию о репозиториях.
- Запускать инструменты, обёрнутые в Gradio и совместимые с MCP.

### Подключение к Hugging Face MCP

1. Получите конфигурацию для своего клиента на [странице интеграции Hugging Face](https://huggingface.co/docs/hub/mcp).
2. Добавьте сервер в конфиг вашего Host (например, Claude Desktop):

```json
{
  "mcpServers": {
    "huggingface": {
      "command": "npx",
      "args": [
        "-y",
        "@huggingface/mcp",
        "--hf-token",
        "YOUR_HF_TOKEN"
      ]
    }
  }
}
```

3. Перезапустите Host.
4. Попросите LLM использовать Hugging Face инструменты, например:  
   *«Найди на Hugging Face модель для перевода с английского на русский»*.

---

## Gradio MCP Integration

**Gradio** — библиотека Python для быстрого создания веб-интерфейсов для ML-моделей. Начиная с версии `gradio[mcp]`, любое Gradio-приложение может автоматически стать MCP-сервером.

### Создание Gradio-приложения с MCP

Установка:
```bash
pip install 'gradio[mcp]'
```

Пример:

```python
# app.py
import gradio as gr

def greet(name, intensity):
    return "Привет, " + name + "!" * int(intensity)

# Создаём интерфейс
demo = gr.Interface(
    fn=greet,
    inputs=["text", "slider"],
    outputs=["text"],
    title="Приветствие"
)

# Запускаем с поддержкой MCP
demo.launch(mcp_server=True)
```

После запуска будут доступны:
- Веб-интерфейс по адресу `http://127.0.0.1:7860`.
- MCP-сервер на том же порту с эндпоинтами `/mcp` (для SSE) и `/mcp/schema` (описание инструментов).

Каждая функция Gradio (например, `greet`) автоматически превращается в **MCP tool**. Название инструмента формируется из имени функции.

### Деплой на Hugging Face Spaces

1. Создайте новый Space на Hugging Face с типом **Gradio**.
2. Загрузите код (например, `app.py` и `requirements.txt` с `gradio[mcp]`).
3. В настройках Space убедитесь, что переменная окружения `GRADIO_MCP_SERVER` установлена в `true` (или используйте `launch(mcp_server=True)` в коде).
4. Space автоматически запустит Gradio с MCP-сервером.

Теперь вы можете подключаться к этому Space как к удалённому MCP-серверу, используя SSE-транспорт и URL вида `https://username-space.hf.space/mcp`.

---

## Заключение

MCP — мощный протокол, который стандартизирует взаимодействие AI-моделей с внешними инструментами и данными. Благодаря поддержке Hugging Face и Gradio, создание и использование MCP-серверов становится простым и доступным.

### Полезные ссылки

- [Официальная документация MCP](https://modelcontextprotocol.io)
- [Hugging Face MCP Integration](https://huggingface.co/docs/hub/mcp)
- [Gradio + MCP](https://www.gradio.app/guides/mcp)
- [MCP Python SDK](https://github.com/modelcontextprotocol/python-sdk)

---